# Notebook 31 — Robust Feature Engineering and Random Forest Signal Optimization

**Research-only.** Nested expanding chronological folds; original final 20% holdout remains closed. No ledger writes. Compare raw features against train-only winsorization (0.5%, 1%, 2%) across price, volume and combined feature families. No target/return clipping. Hyperparameter and entry-threshold selection uses **inner** folds only; **outer** folds estimate development performance. Beware cross-sectional same-day dependence and that historical labels/feature availability require causal review before production.


In [ ]:
from pathlib import Path
import sys
ROOT=Path.cwd().resolve()
if ROOT.name=='notebooks': ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'))
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from IPython.display import display
from stock_ml_dynamic.pooled_models import load_master
from stock_ml_dynamic.robust_rf_research import run_robust_rf, daily_returns, TREATMENTS
sns.set_theme(style='whitegrid')
SEED=42; CAP=60000; TREES=200; COST_BPS=10; MIN_COUNT=150; MIN_COVERAGE=.025
OUTPUT=ROOT/'artifacts'/'notebook31_robust_rf'


## 1. Load master dataset and preserve closed holdout

In [ ]:
paths=[ROOT/'artifacts/master_tables/1d',ROOT/'notebooks/artifacts/master_tables/1d']
source=next((p for p in paths if (p/'master_labeled.parquet').exists()),None)
if source is None: raise FileNotFoundError('Missing 1d master_labeled.parquet')
master,latest,bundle,manifest=load_master(source)
print('Master rows:',len(master),'tickers:',master.symbol.nunique())


## 2. Nested chronological RF search

**72 inner fits** (3 outer × 2 inner × 3 feature families × 4 clipping strategies × 2 RF leaf sizes), plus up to six selected outer refits. Each model uses 200 trees and up to 60,000 sampled training rows. All clipping bounds, imputation and constant-column filtering are learned on the fitting rows only. Thresholds 0.40–0.60 and entry eligibility are chosen on inner folds, not outer folds.


In [ ]:
results,cutoff=run_robust_rf(master,cap=CAP,trees=TREES,seed=SEED,cost_bps=COST_BPS,
    min_count=MIN_COUNT,min_coverage=MIN_COVERAGE,output_dir=OUTPUT)
print('Closed final holdout begins after:',cutoff)
for name,table in results.items(): print(name,table.shape)


## 3. Inner search leaderboard and out-of-fold decisions

In [ ]:
inner=results['inner'];outer=results['outer'];choices=results['selections']
display(inner.groupby(['features','treatment','leaf'],as_index=False).agg(accuracy=('accuracy','mean'),auc=('auc','mean'),mean_net=('mean_net','mean'),min_coverage=('coverage','min')).sort_values('accuracy',ascending=False).head(35))
display(choices);display(outer)


## 4. Does clipping help? Compare against untouched raw features

In [ ]:
summary=inner.groupby(['features','treatment'],as_index=False).agg(accuracy=('accuracy','mean'),auc=('auc','mean'),logloss=('logloss','mean'))
display(summary.sort_values(['features','accuracy'],ascending=[True,False]))
fig,ax=plt.subplots(figsize=(12,5));sns.barplot(data=summary,x='features',y='accuracy',hue='treatment',ax=ax)
ax.set_title('Inner validation accuracy by feature family and outlier policy');plt.show()


## 5. Selected model accuracy, discrimination, calibration and entry quality

In [ ]:
if not outer.empty:
 display(outer.groupby('objective')[['accuracy','auc','logloss','brier_up','coverage','precision','mean_net']].mean())
 fig,axes=plt.subplots(1,2,figsize=(14,5))
 sns.barplot(data=outer,x='fold',y='accuracy',hue='objective',ax=axes[0]);axes[0].set_title('Outer accuracy')
 sns.barplot(data=outer,x='fold',y='mean_net',hue='objective',ax=axes[1]);axes[1].axhline(0,color='black',linewidth=1);axes[1].set_title('Outer selected-entry net return')
 plt.tight_layout();plt.show()


## 6. Feature importance and clipping boundaries

In [ ]:
importance=results['importance'];bounds=results['bounds']
if not importance.empty:
 ranking=importance.groupby(['objective','feature'],as_index=False).importance.mean()
 for objective in ranking.objective.unique():
  top=ranking[ranking.objective==objective].nlargest(25,'importance')
  display(top)
  plt.figure(figsize=(11,8));sns.barplot(data=top,y='feature',x='importance');plt.title('RF impurity importance: '+objective);plt.tight_layout();plt.show()
if not bounds.empty:
 display(bounds.groupby('feature')[['lower','upper']].median().head(30))


## 7. Trading-day concentration, tail risk and cost sensitivity

In [ ]:
trades=results['trades'];daily=daily_returns(trades)
if not trades.empty:
 display(trades.groupby(['objective','fold']).agg(n=('net_return','size'),mean=('net_return','mean'),median=('net_return','median'),p05=('net_return',lambda x:x.quantile(.05)),p95=('net_return',lambda x:x.quantile(.95))))
 display(daily.groupby(['objective','fold']).agg(days=('day','nunique'),mean_day=('mean_net','mean'),worst_day=('mean_net','min')))
 sensitivity=pd.DataFrame([{'objective':obj,'fold':fold,'cost_bps':cost,'mean_net':(group.gross_return-cost/10000).mean()}
   for (obj,fold),group in trades.groupby(['objective','fold']) for cost in (0,5,10,20,35)])
 display(sensitivity)
 sns.lineplot(data=sensitivity,x='cost_bps',y='mean_net',hue='objective',style='fold',markers=True);plt.axhline(0,color='black');plt.show()
else: print('No selected entries: do not infer a profitable policy.')


## 8. Research decision

Check (a) whether clipped features beat raw on **each outer fold**, (b) accuracy versus majority baseline, (c) class-1 entry precision and coverage, (d) mean and worst trading-day returns after costs, (e) feature-importance stability, and (f) whether an entry policy passed the minimum inner-fold trade-count requirement. These diagnostics are exploratory, not a live-trading approval. **Do not reuse the closed final holdout or write to the prospective ledger.**


In [ ]:
if not outer.empty:
 outer=outer.copy();outer['accuracy_lift_pp']=100*(outer.accuracy-outer.majority_baseline)
 display(outer[['fold','objective','features','treatment','leaf','accuracy_lift_pp','auc','entry_count','coverage','precision','mean_net']])
 print('Outer folds with accuracy above baseline:',int((outer.query("objective=='accuracy'").accuracy_lift_pp>0).sum()),'of 3')
print('Saved diagnostic CSVs to:',OUTPUT)
